# AST + VTT Fusion Pipeline

**Approach:**
1. Use AST laughter probabilities as **additional features**
2. Use VTT [laughter] markers as **ground truth labels**
3. Combine with prosody features + WavLM embeddings
4. Train fusion classifier

This leverages AST's AudioSet knowledge while using verified VTT labels.

In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive')

!pip install -q torch torchaudio librosa numpy pandas
!pip install -q transformers accelerate

import numpy as np
import pandas as pd
import json
import os
import librosa
from tqdm import tqdm

BASE = '/content/drive/MyDrive/chuckle_net'

In [ ]:
# === STEP 1: LOAD VTT LABELS (Ground Truth) ===
# Parse VTT files to get [laughter] markers
import re

def parse_vtt_for_laughter(vtt_path):
    """Parse VTT file and extract [laughter] marker positions."""
    laughter_segments = []
    
    try:
        with open(vtt_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Find all timestamps and text
        timestamp_pattern = r'(\d{2}:\d{2}:\d{2}\.\d{3})\s*-->\s*(\d{2}:\d{2}:\d{2}\.\d{3})'
        timestamps = re.findall(timestamp_pattern, content)
        
        # Find [laughter] markers - they appear AFTER timestamps
        # Split by timestamps to get text between them
        segments = re.split(timestamp_pattern, content)
        
        for i in range(2, len(segments), 3):
            if i >= len(segments) - 1:
                break
                
            start_time = parse_time(segments[i-2] if i-2 >= 0 else '00:00:00.000')
            end_time = parse_time(segments[i-1] if i-1 >= 0 else '00:00:00.000')
            text = segments[i] if i < len(segments) else ''
            
            # Check if this segment has [laughter]
            if '[laughter]' in text.lower():
                laughter_segments.append({
                    'start': start_time,
                    'end': end_time,
                    'duration': end_time - start_time
                })
        
    except Exception as e:
        pass
    
    return laughter_segments

def parse_time(time_str):
    """Parse VTT timestamp to seconds."""
    match = re.match(r'(\d{2}):(\d{2}):(\d{2})\.(\d{3})', time_str)
    if match:
        h, m, s, ms = match.groups()
        return int(h)*3600 + int(m)*60 + int(s) + int(ms)/1000
    return 0.0

# Find all VTT files
VTT_DIR = f'{BASE}/vtt'
vtt_files = [f for f in os.listdir(VTT_DIR) if f.endswith('.vtt')]
print(f'Found {len(vtt_files)} VTT files')

# Parse all VTT files
video_laughter = {}
for vtt_file in tqdm(vtt_files, desc='Parsing VTT'):
    video_id = vtt_file.replace('.en.vtt', '').replace('.vtt', '')
    vtt_path = f'{VTT_DIR}/{vtt_file}'
    laughter_segs = parse_vtt_for_laughter(vtt_path)
    if laughter_segs:
        video_laughter[video_id] = laughter_segs

print(f'\nVideos with laughter: {len(video_laughter)}')
print(f'Sample: {list(video_laughter.items())[:2]}')

In [ ]:
# === STEP 2: LOAD AST LABELS (As Features) ===
# Load AST laughter probabilities
import glob

AST_DIR = f'{BASE}/ast_labels'

ast_results = {}
for json_file in glob.glob(f'{AST_DIR}/*.json'):
    with open(json_file) as f:
        data = json.load(f)
        video_id = data['video_id']
        ast_results[video_id] = data

print(f'Loaded AST results for {len(ast_results)} videos')

# Show sample
sample = list(ast_results.items())[0]
print(f"Sample: {sample[0]}, clips: {sample[1]['n_clips']}, max_prob: {sample[1]['max_laugh_prob']:.4f}")

In [ ]:
# === STEP 3: LOAD EXISTING DATA (WavLM embeddings) ===
# Load existing WavLM embeddings and splits

train_data = np.load(f'{BASE}/vtt_splits/train.npz')
valid_data = np.load(f'{BASE}/vtt_splits/valid.npz')
test_data = np.load(f'{BASE}/vtt_splits/test.npz')

X_train = train_data['X']
y_train = train_data['y']
X_valid = valid_data['X']
y_valid = valid_data['y']
X_test = test_data['X']
y_test = test_data['y']

print(f'Train: {X_train.shape}, pos={y_train.sum()} ({100*y_train.mean():.1f}%)')
print(f'Valid: {X_valid.shape}, pos={y_valid.sum()} ({100*y_valid.mean():.1f}%)')
print(f'Test: {X_test.shape}, pos={y_test.sum()} ({100*y_test.mean():.1f}%)')

# Note: These splits have the corrupted labels (1.1% unique positives)
# We'll use them to get features, but use VTT labels for training

In [ ]:
# === STEP 4: BUILD NEW TRAINING DATA ===
# For videos with VTT [laughter], rebuild labels using VTT
# For AST, use probabilities as additional features

def build_dataset_for_video(video_id, ast_data, laughter_segs):
    """Build training samples for one video using VTT labels."""
    clips = ast_data['clips']
    
    samples = []
    for clip in clips:
        clip_start = clip['start']
        clip_end = clip['end']
        ast_prob = clip['laughter_prob']
        
        # Check if clip overlaps with any laughter segment
        is_laughter = False
        for laff_seg in laughter_segs:
            # Overlap if: clip_start < laff_end AND clip_end > laff_start
            if clip_start < laff_seg['end'] and clip_end > laff_seg['start']:
                is_laughter = True
                break
        
        samples.append({
            'video_id': video_id,
            'start': clip_start,
            'end': clip_end,
            'ast_prob': ast_prob,
            'label': 1 if is_laughter else 0
        })
    
    return samples

# Build dataset for videos with both AST and VTT laughter
all_samples = []
matched_videos = 0

for video_id, ast_data in tqdm(ast_results.items(), desc='Building dataset'):
    if video_id in video_laughter:
        laughter_segs = video_laughter[video_id]
        samples = build_dataset_for_video(video_id, ast_data, laughter_segs)
        all_samples.extend(samples)
        matched_videos += 1

print(f'\nMatched {matched_videos} videos with both AST and VTT laughter')
print(f'Total samples: {len(all_samples)}')

if all_samples:
    df = pd.DataFrame(all_samples)
    print(f"Positive: {df['label'].sum()} ({100*df['label'].mean():.1f}%)")
    print(f"AST prob range: {df['ast_prob'].min():.4f} - {df['ast_prob'].max():.4f}")

In [ ]:
# === STEP 5: EXTRACT FEATURES ===
# Extract prosody features + combine with AST probabilities

def extract_prosody_fast(waveform, sr=22050):
    """Extract fast prosody features (no pyin)."""
    features = []
    
    # Energy
    rms = librosa.feature.rms(y=waveform)[0]
    features.extend([
        np.mean(rms), np.std(rms), 
        np.max(rms), np.min(rms),
        np.max(rms) / (np.mean(rms) + 1e-8)
    ])
    
    # ZCR
    zcr = librosa.feature.zero_crossing_rate(y=waveform)[0]
    features.extend([np.mean(zcr), np.std(zcr)])
    
    # MFCCs
    mfccs = librosa.feature.mfcc(y=waveform, sr=sr, n_mfcc=13)
    for i in range(13):
        features.extend([np.mean(mfccs[i]), np.std(mfccs[i])])
    
    # Spectral
    spec_cent = librosa.feature.spectral_centroid(y=waveform, sr=sr)[0]
    spec_bw = librosa.feature.spectral_bandwidth(y=waveform, sr=sr)[0]
    spec_rolloff = librosa.feature.spectral_rolloff(y=waveform, sr=sr)[0]
    features.extend([np.mean(spec_cent), np.std(spec_cent)])
    features.extend([np.mean(spec_bw), np.std(spec_bw)])
    features.extend([np.mean(spec_rolloff), np.std(spec_rolloff)])
    
    # Spectral contrast
    rms_contrast = librosa.feature.spectral_contrast(y=waveform, sr=sr)
    features.extend([np.mean(rms_contrast), np.std(rms_contrast)])
    
    return np.array(features, dtype=np.float32)

print('Extracting features for labeled clips...')

AUDIO_DIR = f'{BASE}/audio'
X_list = []
y_list = []
ast_probs = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc='Extracting'):
    video_id = row['video_id']
    start, end = row['start'], row['end']
    label = row['label']
    
    audio_path = f'{AUDIO_DIR}/{video_id}.m4a'
    if not os.path.exists(audio_path):
        continue
    
    try:
        # Load clip
        y, sr = librosa.load(audio_path, offset=start, duration=end-start, sr=22050, mono=True)
        
        # Extract prosody
        prosody = extract_prosody_fast(y, sr)
        
        X_list.append(prosody)
        y_list.append(label)
        ast_probs.append(row['ast_prob'])
        
    except Exception as e:
        continue

X = np.array(X_list)
y = np.array(y_list)
ast_probs = np.array(ast_probs)

print(f'\nExtracted {len(X)} samples')
print(f'Positive: {y.sum()} ({100*y.mean():.1f}%)')
print(f'AST probs range: {ast_probs.min():.4f} - {ast_probs.max():.4f}')

In [ ]:
# === STEP 6: TRAIN FUSION MODEL ===
# Combine prosody + AST probability as features

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
import torch
import torch.nn as nn

# Create fusion features: [prosody (43-dim) + AST prob (1-dim)]
X_fusion = np.column_stack([X, ast_probs])
print(f'Fusion features: {X_fusion.shape}')

# Split by video for fair evaluation
video_ids = df['video_id'].values[:len(X)]
unique_videos = list(set(video_ids))
train_vids, test_vids = train_test_split(unique_videos, test_size=0.2, random_state=42)

train_mask = np.array([v in train_vids for v in video_ids])
test_mask = np.array([v in test_vids for v in video_ids])

X_train, y_train = X_fusion[train_mask], y[train_mask]
X_test, y_test = X_fusion[test_mask], y[test_mask]

print(f'Train: {len(X_train)}, pos={y_train.sum()} ({100*y_train.mean():.1f}%)')
print(f'Test: {len(X_test)}, pos={y_test.sum()} ({100*y_test.mean():.1f}%)')

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Logistic Regression
print('\nTraining Logistic Regression...')
clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)
f1 = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)

print(f'\n=== LOGISTIC REGRESSION RESULTS ===')
print(f'F1: {f1:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall: {rec:.4f}')

In [ ]:
# === STEP 7: TRAIN MLP ===
# Train MLP on fusion features

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

class FusionMLP(nn.Module):
    def __init__(self, dim_in):
        super().__init__()
        self.bn0 = nn.BatchNorm1d(dim_in)
        self.net = nn.Sequential(
            nn.Linear(dim_in, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        return self.net(self.bn0(x)).squeeze(-1)

X_tr = torch.FloatTensor(X_train_scaled)
y_tr = torch.FloatTensor(y_train)
X_te = torch.FloatTensor(X_test_scaled)
y_te = torch.FloatTensor(y_test)

model = FusionMLP(X_train_scaled.shape[1]).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
pos_w = torch.tensor((1-y_tr.mean())/max(0.01, y_tr.mean())).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)

best_f1 = 0
best_state = None
batch_size = 64

for epoch in range(50):
    model.train()
    for i in range(0, len(X_tr), batch_size):
        bx = X_tr[i:i+batch_size].to(device)
        by = y_tr[i:i+batch_size].to(device)
        opt.zero_grad()
        out = model(bx)
        loss = loss_fn(out, by)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    sched.step()
    
    model.eval()
    with torch.no_grad():
        pred = (torch.sigmoid(model(X_te.to(device))) > 0.5).cpu().numpy().astype(int)
        f1 = f1_score(y_te.numpy(), pred, zero_division=0)
    
    if f1 > best_f1:
        best_f1 = f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}: F1={f1:.4f}')

model.load_state_dict(best_state)
model.eval()

with torch.no_grad():
    pred = (torch.sigmoid(model(X_te.to(device))) > 0.5).cpu().numpy().astype(int)
    f1 = f1_score(y_te.numpy(), pred)
    prec = precision_score(y_te.numpy(), pred, zero_division=0)
    rec = recall_score(y_te.numpy(), pred, zero_division=0)

print(f'\n=== MLP RESULTS ===')
print(f'F1: {f1:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall: {rec:.4f}')

In [ ]:
# === SAVE RESULTS ===
import pickle

results = {
    'n_samples': len(X),
    'n_positive': int(y.sum()),
    'positive_rate': float(y.mean()),
    'lr_f1': float(f1),
    'lr_precision': float(prec),
    'lr_recall': float(rec),
    'mlp_f1': float(best_f1),
}

with open(f'{BASE}/fusion_results.json', 'w') as f:
    json.dump(results, f, indent=2)

torch.save(best_state, f'{BASE}/fusion_model.pt')
torch.save(scaler, f'{BASE}/fusion_scaler.pt')

print(f'Results saved to {BASE}/fusion_results.json')
print(f'\nFinal Results: {results}')